# Week 2 - Python for Data Analysis Refresher

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 2 - Python for Data Analysis Refresher**

---

## Session topics (from the course outline)

- Python basics: variables, loops, functions
- Introduction to the Jupyter notebook
- Installing and using libraries: numpy, pandas, matplotlib

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 1** | Explain the fundamental concepts of data analysis and visualization |
| **CLO 2** | Identify suitable tools and techniques for analyzing different types of datasets |
| **CLO 5** | Perform exploratory data analysis (EDA) to summarize and interpret datasets |

## Reading

| Topic | Source |
|---|---|
| Language semantics, scalar types, control flow | McKinney §2.3 |
| Data structures and sequences | McKinney §3.1 |
| Functions, scope, generators, exceptions | McKinney §3.2 |
| Simple data types | Pfaffelhuber §2 |
| Control flow, functions, exceptions, files | Pfaffelhuber §3 |
| Container data types, comprehensions | Pfaffelhuber §4 |
| Modules and imports | Pfaffelhuber §5.1 |

## Why this session exists

You have met Python before; it is the prerequisite for this course. This session is not a
complete reintroduction. It covers the specific subset of the language that data work leans
on, and the handful of behaviours that cause real bugs in analysis code.

The session ends by doing one aggregation twice: once in plain Python, and once in pandas.
The comparison is the argument for everything that follows in the course.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 - Theory and concepts

## 1.1 Names, objects and assignment

In Python, a variable is a **name bound to an object**, not a box holding a value. The
distinction is invisible until two names refer to the same object.

```python
a = [1, 2, 3]
b = a           # b and a now name the SAME list
b.append(4)
print(a)        # [1, 2, 3, 4]  -- a changed too
```

This is the single most common source of surprise in data code, and it depends on whether
the object is mutable.

| Category | Types | Assignment shares the object? |
|---|---|---|
| **Immutable** | `int`, `float`, `str`, `bool`, `tuple`, `None` | Irrelevant - the object cannot change |
| **Mutable** | `list`, `dict`, `set`, `ndarray`, `DataFrame` | Yes, and changes are visible through every name |

To copy rather than share, construct a new object: `b = a.copy()` or `b = list(a)`.

## 1.2 Scalar types

Pfaffelhuber (§2) and McKinney (§2.3) cover the same small set.

| Type | Example | Notes for data work |
|---|---|---|
| `int` | `42` | Unbounded size in Python; NumPy integers are fixed-width and can overflow |
| `float` | `3.14` | Double precision, and therefore approximate - see §1.6 |
| `bool` | `True` | A subclass of `int`; `True + True` is `2`, which pandas exploits for counting |
| `str` | `"Sophea"` | Immutable; every "modification" builds a new string |
| `NoneType` | `None` | Absence of a value. Test with `is None`, never `== None` |

## 1.3 Control flow

`if` / `elif` / `else`, `for`, `while`, `break`, `continue`. Two points matter for analysis
code.

**Iterate over the thing, not over its indices.** `for row in rows:` rather than
`for i in range(len(rows)):`. When you need the index as well, use `enumerate`:

```python
for i, row in enumerate(rows):
    ...
```

**To walk two sequences together, use `zip`:**

```python
for name, score in zip(names, scores):
    ...
```

`zip` stops at the shorter sequence, silently. If the two are supposed to be the same
length, check it rather than assuming.

## 1.4 Functions

```python
def summarise(values, ndigits: int = 2) -> dict:
    """Return the count and mean of values."""
    return {"n": len(values), "mean": round(sum(values) / len(values), ndigits)}
```

Points that matter:

- **Return several values as a tuple or a dict.** `return a, b` builds a tuple;
  `first, second = summarise(...)` unpacks it. A dict is clearer when there are more than
  two, which is why every function you write this semester returns one.
- **Default arguments are evaluated once**, when the function is defined - not on each call.
  With a mutable default this produces a genuine bug, demonstrated in Part 2.
- **Scope**: a name assigned inside a function is local to it. Reading an outer name works;
  assigning to one creates a new local unless declared `global`. Avoid relying on either.

## 1.5 The four container types

This is the part of the language data work uses most heavily.

| Type | Written | Ordered | Mutable | Duplicates | Lookup cost | Use it for |
|---|---|:---:|:---:|:---:|---|---|
| `list` | `[1, 2, 3]` | yes | yes | yes | $O(n)$ | A sequence you will add to or reorder |
| `tuple` | `(1, 2, 3)` | yes | no | yes | $O(n)$ | A fixed record; a dict key |
| `dict` | `{"a": 1}` | insertion | yes | keys unique | $O(1)$ | Lookup by name; a record |
| `set` | `{1, 2, 3}` | no | yes | no | $O(1)$ | Membership tests; deduplication |

**The lookup cost column is the practical one.** Checking `x in my_list` examines every
element in turn. Checking `x in my_set` computes a hash and looks in one place. For a list
of 10,000 items inside a loop over another 10,000, that difference is the gap between
instant and unusable.

## 1.6 Floating-point comparison

```python
0.1 + 0.2 == 0.3        # False
```

This is not a Python defect. Binary floating point cannot represent 0.1 exactly, so the sum
is `0.30000000000000004`. Every language with IEEE-754 floats behaves this way.

**Never compare floats with `==`.** Use a tolerance:

```python
import numpy as np
np.isclose(0.1 + 0.2, 0.3)      # True
```

Every numerical check in this course's homework uses `np.isclose` for exactly this reason.

## 1.7 Comprehensions

A comprehension builds a container from an iterable in one expression.

```python
squares   = [x**2 for x in range(10)]                 # list
evens     = [x for x in range(10) if x % 2 == 0]      # with a filter
lengths   = {word: len(word) for word in words}       # dict
distinct  = {word.lower() for word in words}          # set
```

The general form is `[expression for item in iterable if condition]`. Prefer a comprehension
when you are building a container from another one; prefer an explicit loop when the body
does several things, because a comprehension carrying three clauses is harder to read than
the loop it replaced.

## 1.8 Exceptions

```python
try:
    value = float(raw)
except ValueError:
    value = None
```

Catch the specific exception you expect. A bare `except:` also catches typing errors in your
own code and turns a loud bug into a silent wrong answer.

Real data files contain empty strings, the text `"N/A"`, numbers with thousands separators
and dates in three formats. Exception handling is how a loader survives the first pass over
them, and it is why Week 5 exists.

## 1.9 Files and the `with` statement

```python
with open("data.csv", "r", encoding="utf-8") as f:
    contents = f.read()
```

The `with` statement closes the file when the block ends, including when an exception is
raised inside it. Always specify `encoding="utf-8"`: the default differs between operating
systems, and a file written on one machine then fails to open on another.

## 1.10 Imports

```python
import numpy as np                    # the community convention
from collections import Counter       # one name from a module
```

The abbreviations `np`, `pd`, `plt` and `sns` are conventions, not requirements, but they are
universal. Use them, because every example you read online does.

---

---

# Part 2 - Instructor code example

The code below is complete and executable. It works through a small dataset three ways:
first with plain Python containers, then with the language features that shorten that work,
and finally with pandas - which is where the rest of the course lives.

## 2.1 A dataset as plain Python

Before pandas, a table is a list of dictionaries: one dictionary per row, keys as column
names.

In [ ]:
def generate_enrolment_records(n_records: int = 60, seed: int = RANDOM_SEED) -> list:
    """Build a small enrolment dataset as a list of dictionaries.

    One dictionary per student. This is the plain-Python representation of a
    table, before pandas is introduced in section 2.5.

    Parameters
    ----------
    n_records : int
        Number of student records to generate.
    seed : int
        Seed for reproducibility.

    Returns
    -------
    list of dict
        Keys: student_id, name, faculty, credits, fee_usd, scholarship.
    """
    generator = np.random.default_rng(seed)

    faculties = ["Engineering", "Business", "Science", "Arts"]
    given_names = ["Sophea", "Dara", "Chanthou", "Rithy", "Bopha", "Sokha",
                   "Vanna", "Pisey", "Mealea", "Kosal"]

    records = []
    for i in range(n_records):
        faculty = generator.choice(faculties, p=[0.35, 0.30, 0.20, 0.15])
        credits = int(generator.choice([12, 15, 18], p=[0.25, 0.50, 0.25]))
        records.append({
            "student_id": f"S{2026000 + i}",
            "name": str(generator.choice(given_names)),
            "faculty": str(faculty),
            "credits": credits,
            "fee_usd": round(credits * 95.0, 2),
            "scholarship": bool(generator.choice([True, False], p=[0.22, 0.78])),
        })
    return records


records = generate_enrolment_records()

print(f"{len(records)} records, each a dict with {len(records[0])} keys\n")
for record in records[:3]:
    print(record)

print("\nThe keys are the column names:")
print(list(records[0].keys()))

## 2.2 Loops, comprehensions and the readable middle ground

The same question - what is the total fee income? - written three ways.

In [3]:
# Version 1: an explicit loop. Correct, and clear about what it does.
total = 0.0
for record in records:
    total += record["fee_usd"]
print(f"Loop        : ${total:,.2f}")

# Version 2: a comprehension feeding sum(). Shorter, and reads as one idea.
total = sum(record["fee_usd"] for record in records)
print(f"Comprehension: ${total:,.2f}")

# Version 3: with a filter. Only students who are not on a scholarship.
paying_total = sum(r["fee_usd"] for r in records if not r["scholarship"])
print(f"Fee-paying  : ${paying_total:,.2f}")

# Counting with a comprehension: booleans are integers, so they sum.
n_scholarship = sum(r["scholarship"] for r in records)
print(f"\nScholarship holders: {n_scholarship} of {len(records)}")
print("This works because True == 1 and False == 0 in Python.")

NameError: name 'PACKAGES' is not defined

In [ ]:
# Where a comprehension stops helping.
# This is legal and hard to read:
awkward = [r["name"].upper() if r["scholarship"] else r["name"].lower()
           for r in records if r["credits"] >= 15 and r["faculty"] != "Arts"][:5]
print("Hard to read:", awkward)

# The same logic as a loop, which most readers prefer:
readable = []
for record in records:
    if record["credits"] < 15 or record["faculty"] == "Arts":
        continue
    name = record["name"]
    readable.append(name.upper() if record["scholarship"] else name.lower())
print("Easier to read:", readable[:5])

print("\nBoth give the same answer:", awkward == readable[:5])
print("Choose the comprehension when it fits on one line and does one thing.")

## 2.3 Two behaviours that cause real bugs

The first is the mutable default argument. The second is float comparison. Both look like
edge cases and both appear in ordinary analysis code.

In [ ]:
# THE BUG: a mutable default argument is created ONCE, when the function is
# defined -- not on each call. Every call then shares the same list.

def add_reading_broken(value, readings=[]):      # <-- do not write this
    """Append a value to a list of readings. Contains a deliberate bug."""
    readings.append(value)
    return readings


print("First call :", add_reading_broken(1))
print("Second call:", add_reading_broken(2))
print("Third call :", add_reading_broken(3))
print("\nThe list persists between calls. It was created once, at definition time.")


# THE FIX: use None as the default and build the list inside the function.
def add_reading(value, readings=None):
    """Append a value to a list of readings, creating a new list if none is given."""
    if readings is None:
        readings = []
    readings.append(value)
    return readings


print("\nFixed version:")
print("First call :", add_reading(1))
print("Second call:", add_reading(2))
print("Third call :", add_reading(3))

In [ ]:
# Floating point: why every numeric check in this course uses np.isclose.
print("0.1 + 0.2       =", 0.1 + 0.2)
print("0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3)
print("np.isclose(...)  :", np.isclose(0.1 + 0.2, 0.3))

# It is not a Python problem. 0.1 has no exact binary representation.
print(f"\n0.1 to 20 decimal places: {0.1:.20f}")

# Where this bites in analysis code: a total that should match a known value.
fees = [r["fee_usd"] for r in records]
recomputed = sum(fees)
expected = sum(r["credits"] * 95.0 for r in records)
print(f"\nrecomputed == expected : {recomputed == expected}")
print(f"np.isclose(...)        : {np.isclose(recomputed, expected)}")
print("\nCompare floats with a tolerance, never with ==.")

## 2.4 Aggregation in plain Python

Grouping records by a key and summarising each group is the single most common operation in
data analysis. Here it is with nothing but the standard library.

In [ ]:
def summarise_by_faculty(rows: list) -> dict:
    """Group records by faculty and compute summary statistics for each.

    This is a hand-written group-by. Section 2.5 does the same thing in pandas.

    Parameters
    ----------
    rows : list of dict
        Records with 'faculty', 'fee_usd' and 'credits' keys.

    Returns
    -------
    dict
        Maps faculty name -> dict of n, total_fees, mean_fees, mean_credits.
    """
    # defaultdict(list) creates an empty list the first time a key is seen,
    # which removes the "if key not in dict" check.
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["faculty"]].append(row)

    summary = {}
    for faculty, group in grouped.items():
        fees = [r["fee_usd"] for r in group]
        credits = [r["credits"] for r in group]
        summary[faculty] = {
            "n": len(group),
            "total_fees": round(sum(fees), 2),
            "mean_fees": round(sum(fees) / len(fees), 2),
            "mean_credits": round(sum(credits) / len(credits), 2),
        }
    return summary


by_faculty = summarise_by_faculty(records)

print(f"{'faculty':<14}{'n':>4}{'total':>12}{'mean fee':>11}{'mean cr':>9}")
print("-" * 50)
for faculty in sorted(by_faculty):
    s = by_faculty[faculty]
    print(f"{faculty:<14}{s['n']:>4}{s['total_fees']:>12,.2f}"
          f"{s['mean_fees']:>11,.2f}{s['mean_credits']:>9.2f}")

print("\nThat is roughly 20 lines of code. Section 2.5 does the same thing in one.")

In [ ]:
# Counter: the standard library's tool for the "how many of each" question.
faculty_counts = Counter(r["faculty"] for r in records)
print("Counter result:", faculty_counts)
print("Most common   :", faculty_counts.most_common(2))

# Sets: membership and deduplication, both in constant time per element.
all_names = [r["name"] for r in records]
distinct_names = set(all_names)
print(f"\n{len(all_names)} names, {len(distinct_names)} distinct")
print("Is 'Sophea' among them?", "Sophea" in distinct_names)

# Why the type matters: membership on a list scans it; on a set it does not.
# For small data the difference is invisible; for large data it is decisive.
big_list = list(range(200_000))
big_set = set(big_list)
print(f"\n'199999 in big_list' scans up to 200,000 items.")
print(f"'199999 in big_set'  computes one hash.")
print("Both give:", 199_999 in big_list, 199_999 in big_set)

## 2.5 The same aggregation in pandas

Everything in §2.4 was written by hand. Here it is again.

In [ ]:
# A list of dicts converts straight into a DataFrame.
df = pd.DataFrame(records)

print("DataFrame from the same records:")
print(df.head())
print(f"\nShape: {df.shape}")

In [ ]:
# The whole of section 2.4, in one statement.
pandas_summary = df.groupby("faculty").agg(
    n=("student_id", "count"),
    total_fees=("fee_usd", "sum"),
    mean_fees=("fee_usd", "mean"),
    mean_credits=("credits", "mean"),
).round(2)

print(pandas_summary)

# Confirm the two approaches agree, rather than assuming they do.
manual = summarise_by_faculty(records)
agree = all(
    manual[faculty]["n"] == pandas_summary.loc[faculty, "n"]
    and np.isclose(manual[faculty]["total_fees"], pandas_summary.loc[faculty, "total_fees"])
    and np.isclose(manual[faculty]["mean_fees"], pandas_summary.loc[faculty, "mean_fees"])
    for faculty in manual
)
print(f"\nHand-written and pandas results agree: {agree}")
print("\n20 lines of Python, or one statement. Both are correct.")
print("The rest of this course is about the second one.")

In [ ]:
# A first chart, to close the loop from Week 1.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

order = pandas_summary.sort_values("total_fees").index

axes[0].barh(order, pandas_summary.loc[order, "total_fees"],
             color=PALETTE[0], edgecolor="white")
axes[0].set_title("Total fee income by faculty")
axes[0].set_xlabel("USD")
axes[0].grid(axis="y", alpha=0)

credit_counts = df["credits"].value_counts().sort_index()
axes[1].bar(credit_counts.index.astype(str), credit_counts.to_numpy(),
            color=PALETTE[2], edgecolor="white")
axes[1].set_title("Students by credit load")
axes[1].set_xlabel("credits enrolled")
axes[1].set_ylabel("number of students")
axes[1].grid(axis="x", alpha=0)

plt.tight_layout()
plt.show()

---

# Part 3 - Guided workshop (in class)

Four tasks. Each has a verification cell; run it before moving on.

### Task 1 - Build a comprehension

**Problem.** Using a single list comprehension, build `high_credit_names`: the names of every
student taking **18 credits**, in upper case.

**Hints**

- The general form is `[expression for item in iterable if condition]`.
- The expression is `record["name"].upper()`; the condition tests `record["credits"]`.

**Expected output.** A list of upper-case names, followed by `Task 1 complete`.

In [6]:
# TODO 1: build the list with a single comprehension.
high_credit_names = [record["name"].upper() for record in records if record["credits"] == 18]

print(f"{len(high_credit_names) if high_credit_names else 0} students take 18 credits")
print(high_credit_names[:8] if high_credit_names else "")


12 students take 18 credits
['SREY HENG', 'MALY LY', 'NARY CHAN', 'LEAKENA LY', 'MALY KEO', 'KUNTHEA HENG', 'SAMBATH KEO', 'MALY LY']


In [7]:
import random

random.seed(42)

first_names = ["Sophea", "Dara", "Vannak", "Srey", "Rithy", "Chantha", "Bopha", "Sokha",
               "Kunthea", "Piseth", "Maly", "Sambath", "Nary", "Visal", "Leakena"]
last_names = ["Chan", "Sok", "Lim", "Heng", "Keo", "Meas", "Ly", "Pich"]
faculties = ["Engineering", "Business", "Design", "Science"]

records = [
    {
        "id": i + 1,
        "name": f"{random.choice(first_names)} {random.choice(last_names)}",
        "faculty": random.choice(faculties),
        "credits": random.choice([12, 15, 16, 18, 18, 21]),
        "fee_usd": round(random.uniform(400, 1500), 2),
    }
    for i in range(40)
]

In [8]:
# Verification for Task 1.
expected = [r["name"].upper() for r in records if r["credits"] == 18]

if high_credit_names is None:
    print("Not complete: `high_credit_names` is still None.")
elif not isinstance(high_credit_names, list):
    print(f"Expected a list, got {type(high_credit_names).__name__}.")
elif high_credit_names != expected:
    print(f"Expected {len(expected)} names, got {len(high_credit_names)}.")
    print("Check the credit threshold and that you called .upper().")
else:
    print("Task 1 complete.")

Task 1 complete.


### Task 2 - Write a function returning a dictionary

**Problem.** Complete `describe_records`, which returns a summary of any list of records.

**Requirements.** The returned dictionary must contain exactly these keys:

| Key | Type | Value |
|---|---|---|
| `n` | `int` | Number of records |
| `total_fees` | `float` | Sum of `fee_usd`, rounded to 2 decimal places |
| `mean_credits` | `float` | Mean of `credits`, rounded to 2 decimal places |
| `faculties` | `list` | Distinct faculty names, **sorted alphabetically** |

**Hints**

- `sum(r["fee_usd"] for r in rows)` totals a field.
- `sorted({r["faculty"] for r in rows})` gives distinct values in order: the set removes
  duplicates, `sorted` returns a list.

**Expected output.** The four values printed, followed by `Task 2 complete`.

In [9]:
import numpy as np
def describe_records(rows):
    n = len(rows)
    if n == 0:
        return {"n": 0, "total_fees": 0.0, "mean_credits": 0.0, "faculties": []}

    total_fees = 0.0
    total_credits = 0
    faculties = set()
    for r in rows:
        if any(k not in r for k in ("fee_usd", "credits", "faculty")):
            raise KeyError("Record missing required key")
        total_fees += float(r["fee_usd"])
        total_credits += r["credits"]
        faculties.add(r["faculty"])

    return {
        "n": n,
        "total_fees": round(total_fees, 2),
        "mean_credits": round(total_credits / n, 2),
        "faculties": sorted(faculties),
    }


In [10]:
# Verification for Task 2.
try:
    result = describe_records(records)
except NotImplementedError:
    print("Not complete: TODO 2 still raises NotImplementedError.")
else:
    expected_keys = {"n", "total_fees", "mean_credits", "faculties"}
    problems = []
    if set(result.keys()) != expected_keys:
        problems.append(f"keys should be {sorted(expected_keys)}")
    else:
        if result["n"] != len(records):
            problems.append("n")
        if not np.isclose(result["total_fees"], round(sum(r["fee_usd"] for r in records), 2)):
            problems.append("total_fees")
        if not np.isclose(result["mean_credits"],
                          round(sum(r["credits"] for r in records) / len(records), 2)):
            problems.append("mean_credits")
        if result["faculties"] != sorted({r["faculty"] for r in records}):
            problems.append("faculties (must be distinct and sorted)")
    if problems:
        print("Check these:", ", ".join(problems))
    else:
        print("Task 2 complete.")

Task 2 complete.


### Task 3 - Group by a different key

**Problem.** Build `by_credits`: a dictionary mapping each credit load (12, 15, 18) to the
**number of students** taking it.

Do it twice - once with a loop and a `defaultdict`, once with `Counter` - and confirm the
two agree.

**Hints**

- `defaultdict(int)` starts every new key at `0`, so `counts[key] += 1` works immediately.
- `Counter(r["credits"] for r in records)` does the whole thing in one line.
- A `Counter` is a `dict` subclass; `dict(counter)` converts it for comparison.

**Expected output.** Two matching dictionaries, followed by `Task 3 complete`.

In [11]:
from collections import defaultdict, Counter

# TODO 3a: count with a loop and a defaultdict.
by_credits_loop = defaultdict(int)
for record in records:
    by_credits_loop[record["credits"]] += 1

# TODO 3b: count the same thing with Counter, in one line.
by_credits_counter = Counter(r["credits"] for r in records)

print("Loop   :", dict(by_credits_loop))
print("Counter:", dict(by_credits_counter) if by_credits_counter else None)

Loop   : {21: 12, 12: 7, 18: 12, 16: 8, 15: 1}
Counter: {21: 12, 12: 7, 18: 12, 16: 8, 15: 1}


In [12]:
# Verification for Task 3.
expected = {}
for record in records:
    expected[record["credits"]] = expected.get(record["credits"], 0) + 1

if not by_credits_loop:
    print("Not complete: `by_credits_loop` is empty. Fill in TODO 3a.")
elif by_credits_counter is None:
    print("Not complete: `by_credits_counter` is still None. Fill in TODO 3b.")
elif dict(by_credits_loop) != expected:
    print("The loop version does not match. Expected:", expected)
elif dict(by_credits_counter) != expected:
    print("The Counter version does not match. Expected:", expected)
else:
    print("Task 3 complete. Both methods agree:", expected)

Task 3 complete. Both methods agree: {21: 12, 12: 7, 18: 12, 16: 8, 15: 1}


### Task 4 - Survive messy input

**Problem.** Complete `parse_fee`, which converts a raw value from a file into a float.

Real files contain empty strings, currency symbols, thousands separators and the text
`"N/A"`. The function must return `None` for anything it cannot convert, and must never
raise.

**Requirements**

| Input | Output |
|---|---|
| `"1425.00"` | `1425.0` |
| `"$1,425.00"` | `1425.0` |
| `1425` | `1425.0` |
| `""`, `"N/A"`, `None`, `"abc"` | `None` |

**Hints**

- Convert to `str` first so that numeric input is handled by the same code path.
- `text.replace("$", "").replace(",", "").strip()` removes the decoration.
- Wrap the `float()` call in `try` / `except (ValueError, TypeError)`.

**Expected output.** Each test input with its parsed value, followed by `Task 4 complete`.

In [14]:
def parse_fee(raw):
    """Convert a raw fee value from a file into a float.

    Parameters
    ----------
    raw : any
        The value as read from the file.

    Returns
    -------
    float or None
        The parsed amount, or None if it cannot be converted.
    """
    if raw is None:
        return None

    text = str(raw).replace("$", "").replace(",", "").strip()
    if text == "" or text.upper() == "N/A":
        return None

    try:
        return float(text)
    except (TypeError, ValueError):
        return None


TEST_INPUTS = ["1425.00", "$1,425.00", 1425, "", "N/A", None, "abc", "  980.5  "]
for raw in TEST_INPUTS:
    print(f"  {raw!r:<14} -> {parse_fee(raw)}")

  '1425.00'      -> 1425.0
  '$1,425.00'    -> 1425.0
  1425           -> 1425.0
  ''             -> None
  'N/A'          -> None
  None           -> None
  'abc'          -> None
  '  980.5  '    -> 980.5


In [15]:
# Verification for Task 4.
try:
    parse_fee("1425.00")
except NotImplementedError:
    print("Not complete: TODO 4 still raises NotImplementedError.")
else:
    checks = [
        ("1425.00", 1425.0), ("$1,425.00", 1425.0), (1425, 1425.0),
        ("  980.5  ", 980.5), ("", None), ("N/A", None), (None, None), ("abc", None),
    ]
    problems = []
    for raw, want in checks:
        try:
            got = parse_fee(raw)
        except Exception as exc:
            problems.append(f"{raw!r} raised {type(exc).__name__} (it must never raise)")
            continue
        if want is None and got is not None:
            problems.append(f"{raw!r} should give None, gave {got!r}")
        elif want is not None and (got is None or not np.isclose(got, want)):
            problems.append(f"{raw!r} should give {want}, gave {got!r}")
    if problems:
        print("Check these:")
        for p in problems:
            print("  -", p)
    else:
        print("Task 4 complete. All inputs handled correctly.")

Task 4 complete. All inputs handled correctly.


---

# Part 4 - Homework

Run **Restart Kernel and Run All Cells** before submitting, then push to your
`puc-data-analysis` repository.

## Problem 1 - A record loader that survives a real file

**Problem.** Write `load_records(rows)`, which takes a list of raw dictionaries as read from
a file and returns a cleaned list plus a report of what went wrong.

**Requirements.** Return a dictionary with exactly these keys:

| Key | Type | Value |
|---|---|---|
| `clean` | `list` | Records where **every** field converted successfully |
| `rejected` | `list` | Records that failed, each as `{"row": <original dict>, "reason": <str>}` |
| `n_clean` | `int` | Length of `clean` |
| `n_rejected` | `int` | Length of `rejected` |

A record is **clean** when:

- `student_id` is a non-empty string after stripping whitespace, and
- `credits` converts to an `int` that is greater than zero, and
- `fee_usd` converts to a `float` that is zero or greater.

Each clean record must be a **new dictionary** with `student_id` stripped, `credits` as
`int`, and `fee_usd` as `float`. Do not modify the input dictionaries.

The `reason` string must name the first field that failed: exactly `"student_id"`,
`"credits"` or `"fee_usd"`, checked in that order.

**Hints**

- Check the fields in the stated order and record the first failure, so `reason` is
  predictable.
- `str(value).strip()` handles both `None` and a padded string.
- `int(float("15.0"))` works; `int("15.0")` raises. Convert through `float` first.
- Build the clean record with a dict literal rather than mutating the input.

In [23]:
# Problem 1 — starter code.

def load_records(rows: list) -> dict:
    """Clean raw student records and report rejected records."""
    clean, rejected = [], []

    for row in rows:
        student_id = str(row.get("student_id") or "").strip()

        if not student_id:
            rejected.append({"row": row, "reason": "student_id"})
            continue

        try:
            credits = int(float(str(row.get("credits")).strip()))
            if credits <= 0:
                raise ValueError("credits must be positive")
        except (ValueError, TypeError):
            rejected.append({"row": row, "reason": "credits"})
            continue

        try:
            fee = float(
                str(row.get("fee_usd"))
                .replace("$", "")
                .replace(",", "")
                .strip()
            )
            if fee < 0:
                raise ValueError("fee must not be negative")
        except (ValueError, TypeError):
            rejected.append({"row": row, "reason": "fee_usd"})
            continue

        clean.append({
            "student_id": student_id,
            "credits": credits,
            "fee_usd": fee
        })

    return {
        "clean": clean,
        "rejected": rejected,
        "n_clean": len(clean),
        "n_rejected": len(rejected)
    }

In [24]:
# Problem 1 — validation. All assertions must pass.
report = load_records(RAW)

assert isinstance(report, dict), "load_records must return a dict"
assert set(report.keys()) == {"clean", "rejected", "n_clean", "n_rejected"}, (
    f"Expected keys clean, rejected, n_clean, n_rejected; got {sorted(report.keys())}"
)

assert report["n_clean"] == len(report["clean"]), "n_clean must match len(clean)"
assert report["n_rejected"] == len(report["rejected"]), "n_rejected must match len(rejected)"
assert report["n_clean"] + report["n_rejected"] == len(RAW), (
    "Every input row must end up in exactly one of the two lists"
)

# Rows 0 and 1 are the only clean ones.
assert report["n_clean"] == 2, f"Expected 2 clean records, got {report['n_clean']}"
assert report["n_rejected"] == 5, f"Expected 5 rejected records, got {report['n_rejected']}"

# Types are converted, and whitespace is stripped.
first, second = report["clean"]
assert first["student_id"] == "S2026001"
assert second["student_id"] == "S2026002", "student_id must be stripped of whitespace"
assert isinstance(first["credits"], int), "credits must be an int"
assert isinstance(first["fee_usd"], float), "fee_usd must be a float"
assert first["credits"] == 15
assert np.isclose(first["fee_usd"], 1425.00)

# The input must not be modified.
assert RAW[1]["student_id"] == " S2026002 ", (
    "The input dictionaries must not be modified; build new ones"
)

# Reasons name the first failing field, in the order student_id, credits, fee_usd.
reasons = [item["reason"] for item in report["rejected"]]
assert reasons == ["student_id", "credits", "fee_usd", "credits", "student_id"], (
    f"Expected reasons in row order ['student_id','credits','fee_usd','credits',"
    f"'student_id'], got {reasons}"
)

# Each rejected entry carries the original row.
for item in report["rejected"]:
    assert set(item.keys()) == {"row", "reason"}, "Each rejection needs 'row' and 'reason'"
    assert item["row"] in RAW, "The rejected row must be the original dict"

# An empty input is not an error.
empty = load_records([])
assert empty["n_clean"] == 0 and empty["n_rejected"] == 0
assert empty["clean"] == [] and empty["rejected"] == []

print("Problem 1: all assertions passed.")

NameError: name 'RAW' is not defined

---

## Problem 2 - A general group-and-summarise function

**Problem.** Write `group_summary(rows, group_key, value_key)`, a reusable version of §2.4
that works on any key. This is the plain-Python ancestor of `df.groupby(...).agg(...)`,
which you meet properly in Week 6.

**Requirements.** Return a dictionary mapping each distinct value of `group_key` to a
dictionary with exactly these keys:

| Key | Type | Value |
|---|---|---|
| `n` | `int` | Number of records in the group |
| `total` | `float` | Sum of `value_key` across the group |
| `mean` | `float` | `total / n` |
| `min` | `float` | Smallest value in the group |
| `max` | `float` | Largest value in the group |

**Requirements on behaviour**

- Records whose `value_key` is `None` are **skipped entirely** — they count towards neither
  `n` nor any statistic.
- A group in which every value was skipped must **not appear** in the result at all.
- Raise `KeyError` if `group_key` or `value_key` is missing from the first record.
- Do not round. The tests compare with `np.isclose`.

**Hints**

- Collect the values per group first, then compute the statistics in a second pass. It is
  easier to reason about than updating running totals.
- `defaultdict(list)` removes the "is this key present yet" check.
- Build the result only from groups whose collected list is non-empty.

In [ ]:
# Problem 2 — starter code.

def group_summary(rows: list, group_key: str, value_key: str) -> dict:
    """Group records by one key and summarise a numeric field.

    Parameters
    ----------
    rows : list of dict
        The records to summarise.
    group_key : str
        The field to group by.
    value_key : str
        The numeric field to summarise.

    Returns
    -------
    dict
        Maps each group value -> {'n', 'total', 'mean', 'min', 'max'}.

    Raises
    ------
    KeyError
        If either key is absent from the first record.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete group_summary()")


result = group_summary(records, "faculty", "fee_usd")
print(f"{'faculty':<14}{'n':>4}{'total':>12}{'mean':>10}{'min':>9}{'max':>9}")
print("-" * 58)
for group in sorted(result):
    s = result[group]
    print(f"{group:<14}{s['n']:>4}{s['total']:>12,.2f}"
          f"{s['mean']:>10,.2f}{s['min']:>9,.2f}{s['max']:>9,.2f}")

In [ ]:
# Problem 2 — validation. All assertions must pass.

SAMPLE = [
    {"team": "A", "score": 10.0},
    {"team": "A", "score": 20.0},
    {"team": "B", "score": 7.0},
    {"team": "A", "score": None},     # skipped
    {"team": "C", "score": None},     # only value in group C -> C must not appear
]

out = group_summary(SAMPLE, "team", "score")

assert isinstance(out, dict), "group_summary must return a dict"
assert set(out.keys()) == {"A", "B"}, (
    f"Group C has no usable values and must be absent. Got groups {sorted(out.keys())}"
)

assert set(out["A"].keys()) == {"n", "total", "mean", "min", "max"}, (
    f"Each group needs keys n, total, mean, min, max; got {sorted(out['A'].keys())}"
)

# Group A: two usable values, 10 and 20. The None is skipped, so n is 2 not 3.
assert out["A"]["n"] == 2, f"None values must not count towards n; got {out['A']['n']}"
assert np.isclose(out["A"]["total"], 30.0)
assert np.isclose(out["A"]["mean"], 15.0)
assert np.isclose(out["A"]["min"], 10.0)
assert np.isclose(out["A"]["max"], 20.0)

# Group B: a single value, so min, max and mean all equal it.
assert out["B"]["n"] == 1
assert np.isclose(out["B"]["mean"], 7.0)
assert np.isclose(out["B"]["min"], 7.0)
assert np.isclose(out["B"]["max"], 7.0)

# mean must be consistent with total and n for every group.
for group, stats in out.items():
    assert np.isclose(stats["mean"], stats["total"] / stats["n"]), (
        f"mean must equal total / n for group {group}"
    )
    assert stats["min"] <= stats["mean"] <= stats["max"], (
        f"mean must lie between min and max for group {group}"
    )

# A missing key raises KeyError rather than returning something wrong.
for bad_group, bad_value in [("nope", "score"), ("team", "nope")]:
    try:
        group_summary(SAMPLE, bad_group, bad_value)
    except KeyError:
        pass
    else:
        raise AssertionError(
            f"group_summary(rows, {bad_group!r}, {bad_value!r}) must raise KeyError"
        )

# It must work on the notebook's own data, and agree with pandas.
by_fac = group_summary(records, "faculty", "fee_usd")
reference = pd.DataFrame(records).groupby("faculty")["fee_usd"].agg(
    ["count", "sum", "mean", "min", "max"]
)
for faculty, stats in by_fac.items():
    assert stats["n"] == reference.loc[faculty, "count"]
    assert np.isclose(stats["total"], reference.loc[faculty, "sum"])
    assert np.isclose(stats["mean"], reference.loc[faculty, "mean"])
    assert np.isclose(stats["min"], reference.loc[faculty, "min"])
    assert np.isclose(stats["max"], reference.loc[faculty, "max"])

# It must also group by a different key without modification.
by_credits = group_summary(records, "credits", "fee_usd")
assert set(by_credits.keys()) == {12, 15, 18}, (
    "The function must work on any key, not just 'faculty'"
)

print("Problem 2: all assertions passed.")

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 3

Week 3 replaces the Python list with the NumPy array, and the loop with vectorised
arithmetic. The `group_summary` function you wrote today is the last time this semester you
will aggregate data by hand.

Before the session, read McKinney §4.1–4.3 and Pfaffelhuber §6.1–6.4.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §2.3, §3.1–3.2.

Pfaffelhuber, P. *Python for Data Analysis* (course notes). §2–5.
Available at `https://pfaffelh.github.io/python_for_data/`.